# Tweet Sentiment Analysis — End-to-End

This notebook rebuilds the project from the raw **Sentiment140** CSV. It covers data inspection, balanced sampling, NLP preprocessing, exploratory analysis, leakage-safe train/test splitting, TF-IDF, Naive Bayes, Logistic Regression, evaluation, inference, and artifact export.

Sentiment140 is binary here: label `0` is negative and label `4` is positive. The models are **not trained to recognize neutral sentiment**.

## 1. Imports and reproducibility
Install dependencies once with `pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import time

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    confusion_matrix, precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_colwidth', 120)

## 2. Paths and configuration
The notebook should be opened from the project directory. Outputs use separate notebook filenames so the working Streamlit application is not overwritten accidentally.

In [ ]:
PROJECT_DIR = Path.cwd()
RAW_CSV = PROJECT_DIR / 'training.1600000.processed.noemoticon.csv'
ARTIFACT_DIR = PROJECT_DIR / 'model_notebook'
PROCESSED_CSV = PROJECT_DIR / 'processed_tweets_notebook.csv'

TEST_SIZE = 0.20
MAX_FEATURES = 60_000
PREDICTION_BATCH_SIZE = 100_000
SAVE_ARTIFACTS = True
SAVE_PROCESSED_CSV = True

if not RAW_CSV.exists():
    raise FileNotFoundError(f'Raw dataset not found: {RAW_CSV}')

ARTIFACT_DIR.mkdir(exist_ok=True)
print('Project:', PROJECT_DIR)
print('Dataset:', RAW_CSV.name)

## 3. Load the raw Sentiment140 dataset
The source file has no header. Its six columns are target, tweet ID, date, query flag, username, and tweet text.

In [ ]:
column_names = ['sentiment', 'id', 'date', 'query', 'user', 'text']
raw_df = pd.read_csv(
    RAW_CSV, encoding='latin-1', header=None, names=column_names
)
print(f'Rows: {len(raw_df):,} | Columns: {raw_df.shape[1]}')
display(raw_df.head())
display(raw_df['sentiment'].value_counts().sort_index().rename('count'))

## 4. Basic data-quality checks

In [ ]:
quality = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing': raw_df.isna().sum(),
    'unique': raw_df.nunique(dropna=True),
})
display(quality)
print('Duplicate tweet texts:', raw_df['text'].duplicated().sum())

## 5. Use the complete labeled dataset
All 1.6 million Sentiment140 tweets are retained. The source dataset contains 800,000 negative and 800,000 positive tweets, so no balancing sample is necessary. The later train/test split remains deterministic because a fixed random seed is used.

In [ ]:
df = raw_df.loc[raw_df['sentiment'].isin([0, 4])].copy().reset_index(drop=True)
del raw_df  # release the second full-data reference before text processing

df['label'] = (df['sentiment'] == 4).astype('int8')
df['sentiment_label'] = df['label'].map({0: 'negative', 1: 'positive'})
display(df['sentiment_label'].value_counts())

## 6. Define the NLP preprocessing function
The same transformation must be used during training and live prediction: normalize contractions, preserve `no`/`nor`/`not`, remove URLs and mentions, retain hashtag words, normalize elongated spelling, keep alphabetic text, remove other English stopwords, and apply Porter stemming. Preserving negation prevents `I don't feel good` from collapsing into `feel good`. `punkt` is not required because this project uses whitespace tokenization.

In [ ]:
try:
    STOP_WORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True, raise_on_error=True)
    STOP_WORDS = set(stopwords.words('english'))
STOP_WORDS -= {'no', 'nor', 'not'}

STEMMER = PorterStemmer()

def clean_tweet(text: str) -> str:
    text = str(text).replace('’', "'")
    text = re.sub(r"\bwon't\b", 'will not', text, flags=re.IGNORECASE)
    text = re.sub(r"\bcan't\b", 'can not', text, flags=re.IGNORECASE)
    text = re.sub(r"n't\b", ' not', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', r' \1 ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
    text = re.sub(r'([a-z])\1{2,}', r'\1\1', text)
    tokens = [
        STEMMER.stem(token) for token in text.split()
        if token not in STOP_WORDS and len(token) > 2
    ]
    return ' '.join(tokens)

examples = pd.Series([
    'I absolutely LOVE this product!!! #Amazing',
    '@support This is the worst update ever. https://example.com',
])
display(pd.DataFrame({'original': examples, 'cleaned': examples.map(clean_tweet)}))

## 7. Clean the working dataset and parse time fields

In [ ]:
start = time.perf_counter()
df['clean_text'] = df['text'].map(clean_tweet)
df['date_parsed'] = pd.to_datetime(
    df['date'], format='%a %b %d %H:%M:%S PDT %Y', errors='coerce'
)
df['year'] = df['date_parsed'].dt.year.fillna(2009).astype(int)
df['month'] = df['date_parsed'].dt.month.fillna(1).astype(int)
df['hour'] = df['date_parsed'].dt.hour.fillna(0).astype(int)
print(f'Cleaning completed in {time.perf_counter() - start:.1f} seconds')
print('Empty cleaned tweets:', df['clean_text'].eq('').sum())
display(df[['text', 'clean_text', 'sentiment_label']].head())

## 8. Exploratory analysis
These plots describe the complete dataset; they do not measure model performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['sentiment_label'].value_counts().plot(
    kind='bar', color=['#2ea043', '#f85149'], ax=axes[0], title='Class distribution'
)
df['hour'].value_counts().sort_index().plot(
    kind='line', marker='o', ax=axes[1], title='Tweets by hour'
)
axes[0].set_ylabel('Tweets')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Tweets')
plt.tight_layout()
plt.show()

In [ ]:
def most_common_words(frame, label, n=15):
    counts = Counter(
        token
        for text in frame.loc[frame['sentiment_label'] == label, 'clean_text']
        for token in text.split()
    )
    return pd.DataFrame(counts.most_common(n), columns=['word', 'count'])

print('Most common positive words')
display(most_common_words(df, 'positive'))
print('Most common negative words')
display(most_common_words(df, 'negative'))

## 9. Split before fitting TF-IDF
This ordering prevents data leakage: the vectorizer learns its vocabulary and inverse-document frequencies from the training text only. Stratification preserves the class ratio.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=df['label'],
)
print(f'Train rows: {len(X_train_text):,}')
print(f'Test rows:  {len(X_test_text):,}')
display(pd.DataFrame({
    'train_ratio': y_train.value_counts(normalize=True).sort_index(),
    'test_ratio': y_test.value_counts(normalize=True).sort_index(),
}))

## 10. Fit TF-IDF on training text only
Unigrams capture individual terms; bigrams capture short phrases such as `not good`.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES, ngram_range=(1, 2), sublinear_tf=True,
    min_df=3, strip_accents='unicode', analyzer='word',
)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)
print('Training matrix:', X_train.shape)
print('Test matrix:', X_test.shape)
print('Vocabulary size:', len(tfidf.vocabulary_))

## 11. Train both classifiers

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(alpha=0.5),
    'Logistic Regression': LogisticRegression(
        C=5.0, max_iter=1000, solver='lbfgs'
    ),
}

train_times = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_times[name] = time.perf_counter() - start
    print(f'{name}: {train_times[name]:.2f} seconds')

## 12. Evaluate on the untouched test set
Accuracy summarizes all predictions. Precision, recall, and F1 expose class-specific behavior. The confusion matrix shows the types of errors.

In [ ]:
predictions = {}
probabilities = {}
rows = []

for name, model in models.items():
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions[name], average='binary', zero_division=0
    )
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, predictions[name]),
        'positive_precision': precision,
        'positive_recall': recall,
        'positive_f1': f1,
        'train_seconds': train_times[name],
    })

# Evaluate the soft-voting output used by the dashboard instead of assuming it helps.
ensemble_probability = np.mean(list(probabilities.values()), axis=0)
ensemble_prediction = (ensemble_probability >= 0.5).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_prediction, average='binary', zero_division=0
)
rows.append({
    'model': 'Soft Voting Ensemble',
    'accuracy': accuracy_score(y_test, ensemble_prediction),
    'positive_precision': precision,
    'positive_recall': recall,
    'positive_f1': f1,
    'train_seconds': sum(train_times.values()),
})

comparison_df = pd.DataFrame(rows).sort_values('accuracy', ascending=False)
display(comparison_df.style.format({
    'accuracy': '{:.2%}', 'positive_precision': '{:.2%}',
    'positive_recall': '{:.2%}', 'positive_f1': '{:.2%}',
    'train_seconds': '{:.2f}',
}))

In [ ]:
for name in models:
    print(f'\n{name}\n' + '-' * len(name))
    print(classification_report(
        y_test, predictions[name], target_names=['negative', 'positive']
    ))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, pred), display_labels=['negative', 'positive']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 13. Test predictions on new text
A probability between 0.40 and 0.60 is displayed as **uncertain**, not neutral. This is an interface rule for a binary classifier, not a third learned class.

In [ ]:
def predict_sentiment(text: str) -> pd.DataFrame:
    cleaned = clean_tweet(text)
    if not cleaned:
        raise ValueError('No usable words remain after preprocessing.')
    vector = tfidf.transform([cleaned])
    result = []
    positive_probs = []
    for name, model in models.items():
        p_pos = float(model.predict_proba(vector)[0, 1])
        positive_probs.append(p_pos)
        label = 'positive' if p_pos >= 0.5 else 'negative'
        result.append({
            'model': name, 'prediction': label,
            'P(positive)': p_pos, 'prediction_confidence': max(p_pos, 1 - p_pos),
        })
    best_name = comparison_df.loc[comparison_df['model'].isin(models), 'model'].iloc[0]
    p_pos = float(models[best_name].predict_proba(vector)[0, 1])
    label = 'positive' if p_pos >= 0.6 else 'negative' if p_pos <= 0.4 else 'uncertain'
    result.append({
        'model': f'Selected: {best_name}', 'prediction': label,
        'P(positive)': p_pos, 'prediction_confidence': max(p_pos, 1 - p_pos),
    })
    return pd.DataFrame(result)

display(predict_sentiment('I love this project; it works very well!'))
display(predict_sentiment('This update is terrible and frustrating.'))

## 14. Edge-case evaluation
This diagnostic set checks common failure modes. Negation and spelling variation have expected binary labels. Mixed sentiment and sarcasm are marked ambiguous because a short isolated sentence may not provide enough context. This table is diagnostic evidence, not a replacement for a labeled test dataset.

In [ ]:
edge_cases = [
    ('I feel good', 'positive', 'plain positive'),
    ("I don't feel good", 'negative', 'contraction and negation'),
    ('This is not bad at all', 'positive', 'negated negative word'),
    ('This is sooo goooood', 'positive', 'elongated spelling'),
    ('I love the design but hate the performance', 'ambiguous', 'mixed sentiment'),
    ('Great, another cancelled train', 'ambiguous', 'sarcasm/context'),
]
edge_rows = []
for text, expected, category in edge_cases:
    prediction = predict_sentiment(text).iloc[-1]
    edge_rows.append({
        'text': text, 'category': category, 'expected': expected,
        'prediction': prediction['prediction'],
        'P(positive)': prediction['P(positive)'],
    })
display(pd.DataFrame(edge_rows).style.format({'P(positive)': '{:.2%}'}))

## 15. Save reproducible artifacts
This cell saves to `model_notebook/`, leaving the application's existing `model/` directory untouched. The vectorizer and classifier must always be deployed together.

In [ ]:
individual_results = comparison_df[comparison_df['model'].isin(models)].copy()
best_model_name = individual_results.iloc[0]['model']
best_model = models[best_model_name]

if SAVE_ARTIFACTS:
    joblib.dump(tfidf, ARTIFACT_DIR / 'tfidf_vectorizer.pkl')
    joblib.dump(models['Naive Bayes'], ARTIFACT_DIR / 'nb_model.pkl')
    joblib.dump(models['Logistic Regression'], ARTIFACT_DIR / 'lr_model.pkl')
    joblib.dump(best_model, ARTIFACT_DIR / 'best_model.pkl')

    metadata = {
        'dataset': 'Sentiment140',
        'sample_size': len(df),
        'test_size': TEST_SIZE,
        'random_state': RANDOM_STATE,
        'best_individual_model': best_model_name,
        'metrics': comparison_df.set_index('model').round(6).to_dict(orient='index'),
    }
    with (ARTIFACT_DIR / 'notebook_results.json').open('w', encoding='utf-8') as file:
        json.dump(metadata, file, indent=2)
    print('Saved artifacts to:', ARTIFACT_DIR)

## 16. Build the dashboard dataset
The following probabilities are computed for all sampled rows, including training rows. They are useful for dashboard exploration, but they are **not independent test results**. Only the held-out metrics above should be reported as model performance.

In [ ]:
if SAVE_PROCESSED_CSV:
    nb_probabilities = np.empty(len(df), dtype=np.float32)
    lr_probabilities = np.empty(len(df), dtype=np.float32)
    for start in range(0, len(df), PREDICTION_BATCH_SIZE):
        end = min(start + PREDICTION_BATCH_SIZE, len(df))
        batch_vectors = tfidf.transform(df['clean_text'].iloc[start:end])
        nb_probabilities[start:end] = models['Naive Bayes'].predict_proba(batch_vectors)[:, 1]
        lr_probabilities[start:end] = models['Logistic Regression'].predict_proba(batch_vectors)[:, 1]
    df['nb_confidence'] = nb_probabilities
    df['lr_confidence'] = lr_probabilities
    df['confidence'] = (df['nb_confidence'] + df['lr_confidence']) / 2

    output_columns = [
        'sentiment_label', 'user', 'text', 'clean_text', 'year', 'month', 'hour',
        'confidence', 'nb_confidence', 'lr_confidence',
    ]
    df[output_columns].to_csv(PROCESSED_CSV, index=False)
    print(f'Saved {len(df):,} rows to {PROCESSED_CSV.name}')

## 17. Where the project shifts from notebook to normal Python files

The notebook is the **learning and experimentation layer**. It keeps explanations, intermediate tables, plots, and model comparisons visible cell by cell.

After the workflow was understood, reusable portions were moved into two normal Python files:

1. **`preprocess.py` — repeatable training pipeline**
   - Starts at dataset loading (Sections 3–5).
   - Reuses `clean_tweet` (Section 6).
   - Performs the split, TF-IDF fitting, training, evaluation, and saving (Sections 9–15).
   - It is a script because these steps should run sequentially and reproducibly without manually executing notebook cells.

2. **`app.py` — Streamlit application and inference layer**
   - Loads the saved CSV, TF-IDF vectorizer, models, and evaluation JSON.
   - Reuses exactly the same `clean_tweet` logic for user input.
   - Contains charts, search, filters, model reports, and the real-time prediction interface.
   - It does not train models because retraining whenever the dashboard opens would be slow and incorrect application design.

The handoff is therefore: **notebook exploration → `preprocess.py` training/export → `app.py` loading/inference/UI**.

## 18. Run order

For learning, run this notebook from top to bottom. For normal application use:

```powershell
python preprocess.py       # only when rebuilding models/data
streamlit run app.py       # start the dashboard
```

The current project already contains trained production artifacts, so `preprocess.py` does not need to run every time.